# MTH-424 Project — RAG on Gutenberg #2347 + NarrativeQA
This notebook is designed to run end-to-end in **Google Colab** with minimal manual intervention.

**What it does:**
- Loads and cleans Gutenberg story **#2347** (The Adventure of the Dying Detective)
- Loads the **NarrativeQA** dataset and auto-selects the QA pairs corresponding to that story
- Builds **two different chunking strategies** (required) and compares them
- Indexes chunks in **Milvus Lite** (local, file-backed)
- Retrieves + (optional) reranks, then generates answers with a small instruct model
- Evaluates with **ROUGE-L** and **BLEU**, prints with **4 decimals**, saves CSV/JSON and a PDF report

If you get any error, copy/paste the last 20 lines of the traceback and I will tell you the exact fix.


In [3]:
!pip -q install --upgrade-strategy only-if-needed \
  "pymilvus[milvus_lite]>=2.4.1" \
  "sentence-transformers>=3.0.0" \
  "datasets>=2.20.0" \
  "evaluate>=0.4.2" \
  "transformers>=4.45.0" \
  "accelerate>=0.33.0" \
  "FlagEmbedding>=1.2.10" \
  "reportlab>=4.0.8" \
  "nltk>=3.8.1" \
  "rouge-score" \
  "sacrebleu" \
  "tqdm>=4.66.4"\
  "numpy<2.1"


In [4]:
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)


True

In [5]:

# ==== 1) Imports, reproducibility, device check ====
import os, json, re, math, time, random
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Torch: 2.9.0+cu126
CUDA available: True
GPU: Tesla T4


In [6]:

# ==== 2) Download + clean Gutenberg #2347 text ====
import requests

GUTENBERG_ID = 2347

# Try multiple stable Gutenberg URLs (some mirrors differ).
GUTENBERG_URLS = [
    f"https://www.gutenberg.org/cache/epub/{GUTENBERG_ID}/pg{GUTENBERG_ID}.txt",
    f"https://www.gutenberg.org/files/{GUTENBERG_ID}/{GUTENBERG_ID}-0.txt",
    f"https://www.gutenberg.org/files/{GUTENBERG_ID}/{GUTENBERG_ID}.txt",
]

def fetch_text(urls: List[str], timeout=30) -> str:
    last_err = None
    for url in urls:
        try:
            r = requests.get(url, timeout=timeout)
            if r.status_code == 200 and len(r.text) > 5000:
                print("Downloaded:", url, "chars:", len(r.text))
                return r.text
            else:
                last_err = RuntimeError(f"{url} returned status={r.status_code}, len={len(r.text)}")
        except Exception as e:
            last_err = e
    raise RuntimeError(f"Failed to download Gutenberg text from all URLs. Last error: {last_err}")

raw = fetch_text(GUTENBERG_URLS)

def strip_gutenberg_header_footer(s: str) -> str:
    # Common markers in Gutenberg plain text
    start = re.search(r"\*\*\*\s*START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", s, flags=re.IGNORECASE|re.DOTALL)
    end   = re.search(r"\*\*\*\s*END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", s, flags=re.IGNORECASE|re.DOTALL)
    if start and end and start.end() < end.start():
        s = s[start.end():end.start()]
    # Normalize whitespace
    s = re.sub(r"\r\n", "\n", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    s = re.sub(r"[ \t]{2,}", " ", s)
    return s.strip()

book_text = strip_gutenberg_header_footer(raw)
# Remove Gutenberg front-matter that leaks into first chunks
needle = "Mrs. Hudson"
idx = book_text.find(needle)
if 0 < idx < 2500:
    book_text = book_text[idx:].strip()

print("Cleaned book chars:", len(book_text))
print("Preview:\n", book_text[:600])


Downloaded: https://www.gutenberg.org/cache/epub/2347/pg2347.txt chars: 52261
Cleaned book chars: 31308
Preview:
 Mrs. Hudson, the landlady of Sherlock Holmes, was a long-suffering
woman. Not only was her first-floor flat invaded at all hours by
throngs of singular and often undesirable characters but her remarkable
lodger showed an eccentricity and irregularity in his life which must
have sorely tried her patience. His incredible untidiness, his
addiction to music at strange hours, his occasional revolver practice
within doors, his weird and often malodorous scientific experiments,
and the atmosphere of violence and danger which hung around him made
him the very worst tenant in London. On the other hand,


In [7]:

# ==== 3) Load NarrativeQA and auto-select QA pairs for Gutenberg #2347 ====
from datasets import load_dataset

# Try standard dataset load patterns.
# The exact config name can vary; we try a few safe options.
LOAD_ATTEMPTS = [
    dict(path="deepmind/narrativeqa", name=None),
    dict(path="deepmind/narrativeqa", name="default"),
]

ds = None
last_err = None
for att in LOAD_ATTEMPTS:
    try:
        if att["name"] is None:
            ds = load_dataset(att["path"])
        else:
            ds = load_dataset(att["path"], att["name"])
        print("Loaded NarrativeQA with splits:", list(ds.keys()))
        break
    except Exception as e:
        last_err = e

if ds is None:
    raise RuntimeError(f"Could not load NarrativeQA. Last error: {last_err}")

# Prefer test split (evaluation). Fall back to validation.
split_name = "test" if "test" in ds else ("validation" if "validation" in ds else list(ds.keys())[0])
qa = ds[split_name]
print("Using split:", split_name, "rows:", len(qa))
print("Features:", qa.features)

import json as _json

def flatten_any(x) -> str:
    try:
        return _json.dumps(x, ensure_ascii=False).lower()
    except Exception:
        return str(x).lower()

import re

# ---- STRICT: only match NarrativeQA rows that actually reference Gutenberg #2347 ----
ID_RE = re.compile(r"(?<!\d)2347(?!\d)")  # match 2347 as a standalone number

def ex_blob(ex) -> str:
    """
    Flatten likely metadata fields into one lowercase blob.
    We intentionally prioritize 'document' because it often contains url/id/title/source.
    """
    parts = []
    for k in ["document", "metadata", "story_id", "document_id", "title", "summary", "story"]:
        if k in ex:
            parts.append(flatten_any(ex[k]))
    return " ".join(parts)

def is_target(ex) -> bool:
    b = ex_blob(ex)
    # must look like Gutenberg AND have strong 2347 signal
    if "gutenberg" not in b:
        return False
    if ("pg2347" in b) or (ID_RE.search(b) is not None):
        return True
    return False

# Filter entire split (can take a minute)
qa_target = qa.filter(is_target)
print("Filtered QA rows (strict Gutenberg 2347 match):", len(qa_target))

# ---- If strict match yields 0, fall back to picking best story_key, but still require 2347 signal ----
def get_story_key(ex) -> str:
    for k in ["story_id", "document_id", "id"]:
        if k in ex and ex[k] is not None:
            return str(ex[k])
    if "document" in ex and isinstance(ex["document"], dict):
        for k in ["id", "document_id", "story_id", "source_id", "gutenberg_id"]:
            if k in ex["document"] and ex["document"][k] is not None:
                return str(ex["document"][k])
    return ""

def strong_2347_signal(ex) -> bool:
    b = ex_blob(ex)
    return ("pg2347" in b) or (ID_RE.search(b) is not None)

if len(qa_target) == 0:
    print("No strict matches. Falling back to best story key among rows with strong 2347 signal...")
    best = {}
    for i in range(len(qa)):
        ex = qa[i]
        if not strong_2347_signal(ex):
            continue
        sk = get_story_key(ex)
        if not sk:
            continue
        # score: prioritize pg2347/url/id-like fields by counting occurrences
        b = ex_blob(ex)
        sc = (10 if "pg2347" in b else 0) + (5 if "gutenberg" in b else 0) + (1 if ID_RE.search(b) else 0)
        if sk not in best or sc > best[sk]:
            best[sk] = sc

    if not best:
        raise RuntimeError(
            "Could not find any NarrativeQA rows with a strong 2347 signal in this split. "
            "Print qa[0] to inspect its structure."
        )

    best_story = max(best.items(), key=lambda kv: kv[1])[0]
    print("Selected story key:", best_story, "score:", best[best_story])
    qa_target = qa.filter(lambda ex: get_story_key(ex) == best_story)
    print("Filtered QA rows (best story key):", len(qa_target))

# ---- Extract (question, reference answer) pairs ----
def get_question(ex) -> str:
    q = ex.get("question", "")
    # sometimes question may be dict-like
    if isinstance(q, dict):
        return str(q.get("text", "")).strip()
    return str(q).strip()

def get_answer(ex) -> str:
    ans = ex.get("answers", "")
    if isinstance(ans, list) and len(ans) > 0:
        a0 = ans[0]
        if isinstance(a0, dict):
            return str(a0.get("text", a0.get("answer", ""))).strip()
        return str(a0).strip()
    if isinstance(ans, dict):
        return str(ans.get("text", ans.get("answer", ""))).strip()
    return str(ans).strip()

pairs = []
for ex in qa_target:
    q = get_question(ex)
    a = get_answer(ex)
    if q and a:
        pairs.append((q, a))

print("Usable QA pairs:", len(pairs))
print("Sample QA:\nQ:", pairs[0][0] if pairs else None, "\nA:", pairs[0][1] if pairs else None)

MAX_EVAL = 80
pairs = pairs[:MAX_EVAL]
print("Evaluation pairs used:", len(pairs))
print(qa[0])




/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

data/train-00007-of-00024.parquet:   0%|          | 0.00/132M [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/3461 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/18 [00:00<?, ?it/s]

Loaded NarrativeQA with splits: ['train', 'test', 'validation']
Using split: test rows: 10557
Features: {'document': {'id': Value('string'), 'kind': Value('string'), 'url': Value('string'), 'file_size': Value('int32'), 'word_count': Value('int32'), 'start': Value('string'), 'end': Value('string'), 'summary': {'text': Value('string'), 'tokens': List(Value('string')), 'url': Value('string'), 'title': Value('string')}, 'text': Value('string')}, 'question': {'text': Value('string'), 'tokens': List(Value('string'))}, 'answers': List({'text': Value('string'), 'tokens': List(Value('string'))})}


Filter:   0%|          | 0/10557 [00:00<?, ? examples/s]

Filtered QA rows (strict Gutenberg 2347 match): 28
Usable QA pairs: 28
Sample QA:
Q: What did Watson believe was wrong with Holmes? 
A: Holmes was sick and dying
Evaluation pairs used: 28
{'document': {'id': '0025577043f5090cd603c6aea60f26e236195594', 'kind': 'movie', 'url': 'http://www.awesomefilm.com/script/pumpupthevolume.html', 'file_size': 54078, 'word_count': 11499, 'start': 'Happy Harry Hardon', 'end': 'by Martin Eaves', 'summary': {'text': ' Mark Hunter (Slater), a high school student in a sleepy suburb of Phoenix, Arizona, starts an FM pirate radio station that broadcasts from the basement of his parents\' house. Mark is a loner, an outsider, whose only outlet for his teenage angst and aggression is his unauthorized radio station. His pirate station\'s theme song is "Everybody Knows" by Leonard Cohen and there are glimpses of cassettes by such alternative musicians as The Jesus and Mary Chain, Camper Van Beethoven, Primal Scream, Soundgarden, Ice-T, Bad Brains, Concrete Blonde

In [19]:

# ==== 4) Two chunking strategies (required) ====
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

def chunk_fixed_chars(text: str, chunk_size: int = 1200, overlap: int = 200) -> List[str]:
    chunks = []
    i = 0
    n = len(text)
    while i < n:
        j = min(n, i + chunk_size)
        chunks.append(text[i:j])
        if j == n:
            break
        i = max(0, j - overlap)
    return [c.strip() for c in chunks if c.strip()]

def chunk_by_sentences(text: str, target_chars: int = 1200, overlap_sents: int = 2) -> List[str]:
    sents = nltk.sent_tokenize(text)
    chunks = []
    cur = []
    cur_len = 0
    for s in sents:
        if cur_len + len(s) + 1 > target_chars and cur:
            chunks.append(" ".join(cur).strip())
            # overlap by last N sentences
            cur = cur[-overlap_sents:] if overlap_sents > 0 else []
            cur_len = sum(len(x) + 1 for x in cur)
        cur.append(s)
        cur_len += len(s) + 1
    if cur:
        chunks.append(" ".join(cur).strip())
    return [c for c in chunks if c]

import re

def chunk_by_paragraphs(text: str, target_chars: int = 1200, overlap_paras: int = 1) -> List[str]:
    paras = [p.strip() for p in re.split(r"\n\s*\n+", text) if p.strip()]
    chunks, cur, cur_len = [], [], 0

    for p in paras:
        if cur_len + len(p) + 2 > target_chars and cur:
            chunks.append("\n\n".join(cur).strip())
            cur = cur[-overlap_paras:] if overlap_paras > 0 else []
            cur_len = sum(len(x) + 2 for x in cur)

        cur.append(p)
        cur_len += len(p) + 2

    if cur:
        chunks.append("\n\n".join(cur).strip())

    return [c for c in chunks if c]



# Hyperparameters (you can tune)
CHUNK_SIZE = 1200
OVERLAP = 200
SENT_OVERLAP = 2
TOP_K = 10          # final context chunks used
RETRIEVE_K = 80     # candidates retrieved before rerank


chunks_A = chunk_fixed_chars(book_text, chunk_size=CHUNK_SIZE, overlap=OVERLAP)
chunks_B = chunk_by_paragraphs(book_text, target_chars=CHUNK_SIZE, overlap_paras=1)


print("Chunker A (fixed chars) chunks:", len(chunks_A), "avg chars:", int(sum(len(c) for c in chunks_A)/max(1,len(chunks_A))))
print("Chunker B (sentence)   chunks:", len(chunks_B), "avg chars:", int(sum(len(c) for c in chunks_B)/max(1,len(chunks_B))))
print("\nSample A chunk:\n", chunks_A[0][:400])
print("\nSample B chunk:\n", chunks_B[0][:400])


Chunker A (fixed chars) chunks: 32 avg chars: 1171
Chunker B (sentence)   chunks: 39 avg chars: 1029

Sample A chunk:
 Mrs. Hudson, the landlady of Sherlock Holmes, was a long-suffering
woman. Not only was her first-floor flat invaded at all hours by
throngs of singular and often undesirable characters but her remarkable
lodger showed an eccentricity and irregularity in his life which must
have sorely tried her patience. His incredible untidiness, his
addiction to music at strange hours, his occasional revolver pr

Sample B chunk:
 Mrs. Hudson, the landlady of Sherlock Holmes, was a long-suffering
woman. Not only was her first-floor flat invaded at all hours by
throngs of singular and often undesirable characters but her remarkable
lodger showed an eccentricity and irregularity in his life which must
have sorely tried her patience. His incredible untidiness, his
addiction to music at strange hours, his occasional revolver pr


In [20]:

# ==== 5) Embeddings ====
from sentence_transformers import SentenceTransformer

EMB_MODEL_NAME = "BAAI/bge-small-en-v1.5"
emb_model = SentenceTransformer(EMB_MODEL_NAME, device="cuda" if torch.cuda.is_available() else "cpu")

QUERY_PREFIX = "Represent this question for retrieving supporting passages: "
PASSAGE_PREFIX = "Represent this passage for retrieval: "

def embed_texts(texts: List[str], is_query: bool, batch_size: int = 64) -> np.ndarray:
    if is_query:
        texts = [QUERY_PREFIX + t for t in texts]
    else:
        texts = [PASSAGE_PREFIX + t for t in texts]
    vecs = emb_model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    return np.asarray(vecs, dtype=np.float32)


emb_A = embed_texts(chunks_A, is_query=False)
emb_B = embed_texts(chunks_B, is_query=False)


print("Embedding shapes:", emb_A.shape, emb_B.shape)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shapes: (32, 384) (39, 384)


In [21]:

# ==== 6) Milvus Lite index (local file) ====
from pymilvus import MilvusClient

DB_PATH = "/content/milvus_rag.db"  # file-backed local db in Colab runtime
client = MilvusClient(DB_PATH)

def build_collection(collection_name: str, vectors: np.ndarray, chunks: List[str]):
    # Drop if exists (repeatable runs)
    if client.has_collection(collection_name):
        client.drop_collection(collection_name)

    dim = vectors.shape[1]
    client.create_collection(
        collection_name=collection_name,
        dimension=dim,
        metric_type="IP",  # inner product; works with normalized embeddings (cosine-like)
        consistency_level="Strong",
    )
    # Insert
    data = [
        {"id": i, "vector": vectors[i].tolist(), "text": chunks[i]}
        for i in range(len(chunks))
    ]
    client.insert(collection_name=collection_name, data=data)
    client.flush(collection_name)
    print(f"Built collection '{collection_name}' with {len(chunks)} vectors, dim={dim}")

build_collection("chunks_A", emb_A, chunks_A)
build_collection("chunks_B", emb_B, chunks_B)


Built collection 'chunks_A' with 32 vectors, dim=384
Built collection 'chunks_B' with 39 vectors, dim=384


In [22]:

# ==== 7) Retrieval (+ optional reranking) ====
# Reranker is optional; if it fails to download on your runtime, we continue without it.

RERANK_ENABLED = True
RERANK_MODEL_NAME = "BAAI/bge-reranker-v2-m3"

reranker = None
if RERANK_ENABLED:
    try:
        from FlagEmbedding import FlagReranker
        reranker = FlagReranker(RERANK_MODEL_NAME, use_fp16=torch.cuda.is_available())
        print("Reranker loaded:", RERANK_MODEL_NAME)
    except Exception as e:
        print("Reranker could not be loaded; continuing without reranking.\nError:", e)
        reranker = None

def retrieve(collection: str, query: str, top_k: int = RETRIEVE_K) -> List[Tuple[int, str, float]]:
    # Encode query (keep your existing emb_model; if you already use prefixes, keep that)
    qvec = emb_model.encode([query], normalize_embeddings=True)

    res = client.search(
        collection_name=collection,
        data=qvec,
        limit=top_k,
        output_fields=["text"],  # id is usually returned even if not listed; we handle both cases below
    )

    hits = res[0]
    out = []
    for h in hits:
        ent = h.get("entity", {})
        # Different MilvusClient versions expose id differently:
        cid = ent.get("id", None)
        if cid is None:
            cid = h.get("id", None)
        if cid is None:
            # fallback: try parsing from entity keys
            cid = int(ent.get("pk", 0)) if "pk" in ent else 0

        out.append((int(cid), ent.get("text", ""), float(h.get("distance", 0.0))))
    return out


def rerank(query: str, passages: List[Tuple[int, str, float]]) -> List[Tuple[int, str, float]]:
    if reranker is None or len(passages) <= 1:
        return passages[:TOP_K]

    pairs = [[query, p[1]] for p in passages]  # use only the text for scoring
    scores = reranker.compute_score(pairs)
    order = np.argsort(-np.asarray(scores))
    ranked = [passages[i] for i in order]
    return ranked[:TOP_K]


def build_context(passages: List[str], max_chars: int = 8000) -> str:
    out = []
    total = 0
    for p in passages:
        if total + len(p) > max_chars:
            break
        out.append(p)
        total += len(p)
    return "\n\n".join(out)

# quick sanity check
q0 = pairs[0][0] if pairs else "What is the story about?"
for col in ["chunks_A", "chunks_B"]:
    hits = retrieve(col, q0, top_k=RETRIEVE_K)        # returns (id, text, score)
    hits = rerank(q0, hits)                           # returns top TOP_K hits
    passages = [text for (_, text, _) in hits]        # extract only text
    ctx = build_context(passages)

    print("\nCollection:", col, "| context chars:", len(ctx))
    print(ctx[:600])


Reranker loaded: BAAI/bge-reranker-v2-m3


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.



Collection: chunks_A | context chars: 7207
man is but a child,
and so I will treat you. Whether you like it or not, I will examine
your symptoms and treat you for them."

He looked at me with venomous eyes.

"If I am to have a doctor whether I will or not, let me at least have
someone in whom I have confidence," said he.

"Then you have none in me?"

"In your friendship, certainly. But facts are facts, Watson, and,
after all, you are only a general practitioner with very limited
experience and mediocre qualifications. It is painful to have to say
these things, but you leave me no choice."

I was bitterly hurt.

"Such a remark is unwort

Collection: chunks_B | context chars: 7996
"Possibly not, my good Watson. It may surprise you to know that the
man upon earth who is best versed in this disease is not a medical man,
but a planter. Mr. Culverton Smith is a well-known resident of
Sumatra, now visiting London. An outbreak of the disease upon his
plantation, which was distant from medical

In [23]:

# ==== 8) Generator model (small instruct) ====
from transformers import AutoTokenizer, AutoModelForCausalLM

GEN_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL)
gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

def generate_answer(question: str, context: str, max_new_tokens: int = 80) -> str:
    system = (
        "You are an extractive QA system. "
        "Answer using ONLY exact phrases copied from the provided CONTEXT. "
        "Do NOT paraphrase and do NOT add new words. "
        "Keep the answer to one short sentence or a short phrase. "
        'If not found, reply exactly: Not found in the provided context.'
    )
    user = (
        f"CONTEXT:\n{context}\n\n"
        f"QUESTION:\n{question}\n\n"
        "Answer (copy exact words from context):"
    )

    prompt = (
        f"<|im_start|>system\n{system}<|im_end|>\n"
        f"<|im_start|>user\n{user}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(gen_model.device)
    with torch.no_grad():
        out = gen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode ONLY newly generated tokens (prevents prompt leakage)
    gen_tokens = out[0][inputs["input_ids"].shape[1]:]
    txt = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    return txt


# quick test
test_q = pairs[0][0] if pairs else "Who is the detective?"
hits = retrieve("chunks_A", test_q, top_k=RETRIEVE_K)
hits = rerank(test_q, hits)
ctxA = build_context([text for (_, text, _) in hits])

print(generate_answer(test_q, ctxA))


That Holmes was mistaken


In [24]:

# ==== 9) Run end-to-end QA for both chunkers ====
def run_system(collection_name: str, pairs: List[Tuple[str,str]]) -> List[Dict[str,Any]]:
    outputs = []
    for (q, gold) in tqdm(pairs, desc=f"QA: {collection_name}"):
        hits = retrieve(collection_name, q, top_k=RETRIEVE_K)   # retrieve more
        hits = rerank(q, hits)                                 # rerank down to TOP_K (<=10)
        passages = [t for (_, t, _) in hits]                   # keep texts only for context
        context = build_context(passages)

        pred = generate_answer(q, context)
        outputs.append({"question": q, "gold": gold, "pred": pred})
    return outputs

out_A = run_system("chunks_A", pairs)
out_B = run_system("chunks_B", pairs)

print("Done. Samples:")
print("A pred:", out_A[0]["pred"] if out_A else None)
print("B pred:", out_B[0]["pred"] if out_B else None)


QA: chunks_A:   0%|          | 0/28 [00:00<?, ?it/s]

QA: chunks_B:   0%|          | 0/28 [00:00<?, ?it/s]

Done. Samples:
A pred: That Holmes was mistaken
B pred: That he was overreacting to the situation


In [25]:

# ==== 10) Evaluate (ROUGE-L + BLEU) with 4 decimals ====
import evaluate
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

rouge_metric = evaluate.load("rouge")

def eval_outputs(outputs: List[Dict[str,Any]]) -> Dict[str,float]:
    preds = [o["pred"].strip() for o in outputs]
    refs  = [o["gold"].strip() for o in outputs]

    rouge = rouge_metric.compute(predictions=preds, references=refs, use_stemmer=True)

    smooth = SmoothingFunction().method1
    # corpus_bleu expects: list of samples, each sample = list of reference token lists
    references_tok = [[r.split()] for r in refs]      # [[tokens], [tokens], ...]
    hypotheses_tok = [p.split() for p in preds]       # [tokens, tokens, ...]
    bleu_score = corpus_bleu(references_tok, hypotheses_tok, smoothing_function=smooth)

    return {"rougeL": float(rouge["rougeL"]), "bleu": float(bleu_score)}



scores_A = eval_outputs(out_A)
scores_B = eval_outputs(out_B)

print(f"Chunker A -> ROUGE-L: {scores_A['rougeL']:.4f} | BLEU: {scores_A['bleu']:.4f}")
print(f"Chunker B -> ROUGE-L: {scores_B['rougeL']:.4f} | BLEU: {scores_B['bleu']:.4f}")

# Save results
os.makedirs("/content/outputs", exist_ok=True)
with open("/content/outputs/results.json", "w", encoding="utf-8") as f:
    json.dump({"A": scores_A, "B": scores_B}, f, indent=2)

pd.DataFrame(out_A).to_csv("/content/outputs/predictions_chunker_A.csv", index=False)
pd.DataFrame(out_B).to_csv("/content/outputs/predictions_chunker_B.csv", index=False)

comparison = pd.DataFrame([
    {"chunker": "A_fixed_chars", **scores_A, "chunk_size": CHUNK_SIZE, "overlap": OVERLAP, "top_k": TOP_K, "rerank": reranker is not None},
    {"chunker": "B_sentence",    **scores_B, "chunk_size": CHUNK_SIZE, "overlap_sents": SENT_OVERLAP, "top_k": TOP_K, "rerank": reranker is not None},
])
comparison.to_csv("/content/outputs/results_comparison.csv", index=False)
comparison


Chunker A -> ROUGE-L: 0.1956 | BLEU: 0.0040
Chunker B -> ROUGE-L: 0.2391 | BLEU: 0.0306


,chunker,rougeL,bleu,chunk_size,overlap,top_k,rerank,overlap_sents
0,A_fixed_chars,0.195599,0.004041,1200,200.0,10,True,NaN
1,B_sentence,0.239065,0.030603,1200,NaN,10,True,2.0


In [27]:

# ==== 11) Generate a simple PDF report (auto-filled) ====
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

REPORT_PATH = "/content/outputs/MTH-424_Report.pdf"

c = canvas.Canvas(REPORT_PATH, pagesize=letter)
width, height = letter
y = height - 60

def line(txt, dy=16, font="Helvetica", size=11):
    global y
    c.setFont(font, size)
    c.drawString(60, y, txt)
    y -= dy

line("MTH-424 Project Report — RAG Evaluation", font="Helvetica-Bold", size=14, dy=22)
line(f"Date: {time.strftime('%Y-%m-%d')}")
line(f"Runtime: {'GPU' if torch.cuda.is_available() else 'CPU'}; Torch={torch.__version__}")
line("")
line("Models:", font="Helvetica-Bold")
line(f"Embedding: {EMB_MODEL_NAME}")
line(f"Generator: {GEN_MODEL}")
line(f"Reranker: {RERANK_MODEL_NAME if reranker is not None else 'Disabled/Unavailable'}")
line("")
line("Experiment Parameters:", font="Helvetica-Bold")
line(f"Chunk size (chars): {CHUNK_SIZE}")
line(f"Overlap (A chars): {OVERLAP}")
line(f"Sentence overlap (B sents): {SENT_OVERLAP}")
line(f"Top-K retrieval: {TOP_K}")
line("")
line("Results (printed with 4 decimals):", font="Helvetica-Bold")
line(f"Chunker A (fixed chars): ROUGE-L={scores_A['rougeL']:.4f} | BLEU={scores_A['bleu']:.4f}")
line(f"Chunker B (sentence):   ROUGE-L={scores_B['rougeL']:.4f} | BLEU={scores_B['bleu']:.4f}")
line("")
better = "A" if (scores_A["rougeL"] + scores_A["bleu"]) >= (scores_B["rougeL"] + scores_B["bleu"]) else "B"
line(f"Best (avg of ROUGE-L and BLEU): Chunker {better}", font="Helvetica-Bold")
line("")
line("Discussion (fill if needed):", font="Helvetica-Bold")
line("- Chunking trade-offs observed:")
line("- Retrieval relevance and failure modes:")
line("- What to improve next (top_k, overlap, reranker, prompt):")

c.showPage()
c.save()
print("Saved:", REPORT_PATH)


Saved: /content/outputs/MTH-424_Report.pdf


In [28]:

# ==== 12) Download outputs from Colab ====
from google.colab import files
files.download("/content/outputs/MTH-424_Report.pdf")
files.download("/content/outputs/results_comparison.csv")
files.download("/content/outputs/results.json")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>